
# CO example: distances, Gram matrix, centering, and low-dimensional embedding

This notebook summarizes the full toy example we built for three 2D frames of a CO molecule.

We will show:

1. the three CO frames  
2. the data matrix $X$  
3. the dissimilarity matrix $D$ computed directly  
4. the Gram matrix $K$ computed directly  
5. the reconstruction of $D$ from $K$  
6. the centering matrix $J$  
7. the centered matrix $X_c = JX$  
8. a plot of the original and centered frames  
9. the centered Gram matrix obtained both as $X_c X_c^T$ and as $-\tfrac12 J D J$  
10. the low-dimensional coordinates $Y$ obtained from eigenvalues and eigenvectors


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)



## 1. Three CO frames

We work in 2D and encode each frame as

\[
X_i = (x_C,\; y_C,\; x_O,\; y_O).
\]

The three frames are:

- Frame 1: $C=(-0.6,0)$, $O=(0.5,0)$
- Frame 2: $C=(0,0)$, $O=(1.2,0)$
- Frame 3: $C=(0,-0.2)$, $O=(0,0.8)$


In [ ]:

frames = [
    {"name": "Frame 1", "C": np.array([-0.6,  0.0]), "O": np.array([ 0.5,  0.0])},
    {"name": "Frame 2", "C": np.array([ 0.0,  0.0]), "O": np.array([ 1.2,  0.0])},
    {"name": "Frame 3", "C": np.array([ 0.0, -0.2]), "O": np.array([ 0.0,  0.8])},
]

for f in frames:
    print(f"{f['name']}: C = {tuple(f['C'])}, O = {tuple(f['O'])}")



## 2. Data matrix $X$

Collecting the frame vectors as rows gives

\[
X =
\begin{pmatrix}
-0.6 & 0 & 0.5 & 0 \\
0 & 0 & 1.2 & 0 \\
0 & -0.2 & 0 & 0.8
\end{pmatrix}.
\]


In [ ]:

X = np.array([
    [-0.6,  0.0,  0.5,  0.0],
    [ 0.0,  0.0,  1.2,  0.0],
    [ 0.0, -0.2,  0.0,  0.8],
], dtype=float)

print("X =")
print(X)



## 3. Dissimilarity matrix $D$ computed directly

We use squared Euclidean distances:

\[
D_{ij} = \|X_i - X_j\|^2.
\]


In [ ]:

D = np.sum((X[:, None, :] - X[None, :, :])**2, axis=2)

print("D =")
print(D)



## 4. Gram matrix $K$ computed directly

The Gram matrix contains scalar products:

\[
K_{ij} = X_i \cdot X_j.
\]

So in matrix form,

\[
K = X X^T.
\]


In [ ]:

K = X @ X.T

print("K =")
print(K)



## 5. Reconstruct the dissimilarity matrix from the Gram matrix

Using

\[
D_{ij} = K_{ii} + K_{jj} - 2K_{ij},
\]

we can rebuild the same dissimilarity matrix from $K$.


In [ ]:

diagK = np.diag(K)
D_from_K = diagK[:, None] + diagK[None, :] - 2 * K

print("D reconstructed from K =")
print(D_from_K)
print()
print("D and D_from_K are equal:", np.allclose(D, D_from_K))



## 6. Centering matrix $J$

For $M=3$ frames,

\[
J = I - \frac{1}{3}\,\mathbf{1}\mathbf{1}^T,
\]

where

\[
I =
\begin{pmatrix}
1 & 0 & 0\\
0 & 1 & 0\\
0 & 0 & 1
\end{pmatrix},
\qquad
\mathbf{1} =
\begin{pmatrix}
1\\
1\\
1
\end{pmatrix}.
\]

The product $\mathbf{1}\mathbf{1}^T$ is the $3\times 3$ all-ones matrix.


In [ ]:

M = X.shape[0]
I = np.eye(M)
one = np.ones((M, 1))
J = I - (1/M) * (one @ one.T)

print("I =")
print(I)
print()
print("1 column vector =")
print(one)
print()
print("1 1^T =")
print(one @ one.T)
print()
print("J =")
print(J)



## 7. Centered data matrix $X_c = JX$

Applying $J$ subtracts the average frame from every row of $X$.


In [ ]:

X_avg = X.mean(axis=0)
Xc = J @ X

print("Average frame X_avg =")
print(X_avg)
print()
print("Xc = JX =")
print(Xc)
print()
print("Column means of Xc =")
print(Xc.mean(axis=0))



## 8. Plot of the original and centered frames

The top row shows the original frames.

The bottom row shows the centered frames, that is, the deviations from the average frame:

\[
X_i - X_{\mathrm{avg}}.
\]

These centered frames are **not** physical laboratory coordinates anymore; they are deviations from the mean descriptor.


In [ ]:

C_mean = X_avg[:2]
O_mean = X_avg[2:]

centered_frames = []
for f in frames:
    centered_frames.append({
        "name": f["name"],
        "C": f["C"] - C_mean,
        "O": f["O"] - O_mean
    })

fig, ax = plt.subplots(figsize=(12, 7))

x_offsets = [0, 3.2, 6.4]
y_top = 2.2
y_bottom = -2.2

xmin_local, xmax_local = -1.4, 1.4
ymin_local, ymax_local = -1.1, 1.1

def draw_cell_border(x0, y0):
    rect_x = [x0 + xmin_local, x0 + xmax_local, x0 + xmax_local, x0 + xmin_local, x0 + xmin_local]
    rect_y = [y0 + ymin_local, y0 + ymin_local, y0 + ymax_local, y0 + ymax_local, y0 + ymin_local]
    ax.plot(rect_x, rect_y, lw=1)

def draw_local_axes(x0, y0):
    ax.plot([x0 + xmin_local, x0 + xmax_local], [y0, y0], lw=0.8)
    ax.plot([x0, x0], [y0 + ymin_local, y0 + ymax_local], lw=0.8)

def draw_molecule(x0, y0, C, O, title):
    draw_cell_border(x0, y0)
    draw_local_axes(x0, y0)
    ax.plot([x0 + C[0], x0 + O[0]], [y0 + C[1], y0 + O[1]], lw=2)
    ax.scatter([x0 + C[0], x0 + O[0]], [y0 + C[1], y0 + O[1]], s=120)
    ax.text(x0 + C[0], y0 + C[1] + 0.12, "C", ha="center", va="bottom", fontsize=11)
    ax.text(x0 + O[0], y0 + O[1] + 0.12, "O", ha="center", va="bottom", fontsize=11)
    ax.text(x0, y0 + ymax_local + 0.22, title, ha="center", va="bottom", fontsize=12)

for x0, f in zip(x_offsets, frames):
    draw_molecule(x0, y_top, f["C"], f["O"], f'{f["name"]} (original)')

for x0, f in zip(x_offsets, centered_frames):
    draw_molecule(x0, y_bottom, f["C"], f["O"], f'{f["name"]} (centered)')

ax.text(-2.2, y_top, "Before centering", rotation=90, va="center", ha="center", fontsize=13)
ax.text(-2.2, y_bottom, "After centering", rotation=90, va="center", ha="center", fontsize=13)

ax.text(9.2, 1.2,
        "Mean frame\n"
        f"C̄ = ({C_mean[0]:.3f}, {C_mean[1]:.3f})\n"
        f"Ō = ({O_mean[0]:.3f}, {O_mean[1]:.3f})",
        fontsize=12, va="top")

ax.text(9.2, -1.2,
        "Centered = deviation\nfrom the mean frame\n"
        "Each lower panel shows\nXi − Xavg",
        fontsize=12, va="top")

ax.set_aspect("equal")
ax.set_xlim(-2.8, 11.5)
ax.set_ylim(-4.0, 4.0)
ax.axis("off")
plt.show()



## 9. Centered Gram matrix

We compute the centered Gram matrix in two equivalent ways:

\[
K_c = X_c X_c^T
\]

and

\[
K_c = -\frac12 J D J.
\]

These two formulas should give the same matrix.


In [ ]:

Kc_from_Xc = Xc @ Xc.T
Kc_from_D = -0.5 * J @ D @ J

print("Kc = Xc Xc^T =")
print(Kc_from_Xc)
print()
print("Kc = -1/2 J D J =")
print(Kc_from_D)
print()
print("The two Kc matrices are equal:", np.allclose(Kc_from_Xc, Kc_from_D))



## 10. Eigenvalues, eigenvectors, and low-dimensional coordinates $Y$

We diagonalize the centered Gram matrix:

\[
K_c = V \Lambda V^T.
\]

Then the low-dimensional coordinates are

\[
Y = V \Lambda^{1/2}.
\]

Only the positive eigenvalues are kept.


In [ ]:

eigvals, eigvecs = np.linalg.eigh(Kc_from_Xc)

idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

print("Eigenvalues =")
print(eigvals)
print()
print("Eigenvectors V =")
print(eigvecs)


In [ ]:

tol = 1e-12
positive = eigvals > tol

Lambda_pos = np.diag(eigvals[positive])
V_pos = eigvecs[:, positive]
Y = V_pos @ np.sqrt(Lambda_pos)

print("Positive eigenvalues =")
print(eigvals[positive])
print()
print("Y =")
print(Y)
print()
print("Check Kc = Y Y^T:", np.allclose(Kc_from_Xc, Y @ Y.T))



## Final remark

This is the core idea behind **classical multidimensional scaling**.

Starting from pairwise Euclidean distances between frames, we reconstruct the centered Gram matrix, diagonalize it, and obtain a low-dimensional representation of the trajectory.

For Euclidean data, this is closely related to PCA.
